# Amazon ML Challenge 2026: Business Entity Resolution Pipeline
### End-to-End Execution & Cloud Setup Guide for AWS SageMaker (`ml.c5.18xlarge`)

**Team Name:** unwanted  
**Target Competition Metric:** Official Macro $F_{0.5} \ge 0.9000+$  

---
### ⚙️ SageMaker Cloud Infrastructure & Space Provisioning
To reproduce this solution without disk or memory constraints, provision the environment as follows:

1. **Instance Type:**
   - **Recommended:** `ml.c5.18xlarge` (72 vCPUs, 144 GB RAM, network-optimized).
   - **Alternative:** `ml.m5.16xlarge` (64 vCPUs, 256 GB RAM) or `ml.c5.9xlarge` (36 vCPUs, 72 GB RAM).

2. **EBS Storage Allocation (Crucial — Avoid Disk-Full Failures):**
   - **Required Volume Size:** At least **150 GB to 200 GB** (EBS gp3, 3,000+ IOPS, 125+ MB/s).
   - **Why?** Uncompressed TSV datasets require ~15 GB, the SQLite B-Tree search index (`index.db`) requires ~14 GB, test index (`index_test.db`) requires ~12 GB, and feature vectors require ~10 GB.
   - **How to create/expand space:**
     - *When creating instance:* In AWS SageMaker Console $\rightarrow$ *Notebook instances* $\rightarrow$ *Create notebook instance* $\rightarrow$ Set **Volume size in GB = 200**.
     - *On existing instance:* Stop notebook instance $\rightarrow$ Select *Update* $\rightarrow$ Increase volume size to 200 GB $\rightarrow$ Start instance.
     - Verify free space in terminal with `df -h /`.

3. **Zero-RAM Disk-Indexed Architecture:**
   - All 10.1M records in Source 2 and Source 3 are indexed directly into SQLite B-Tree tables on disk.
   - Uses 36 parallel worker processes (safe deadlock-free sweet spot) to maximize CPU utilization without SQLite reader lock contention.
   - Checkpoints are saved after every chunk to `output/train/lgb_model.pkl` and `output/lgb_model.pkl`.
   - Results are automatically validated against the official competition validator (`validate_submission.py`) and packaged into `unwanted_submission.zip`.


--- 
## 1. Environment & Dependency Setup
Creates a Python virtual environment (`.venv`) and installs project dependencies.

In [ ]:
%%bash
# Dynamic navigation to project root
if [ -f "src/main.py" ]; then
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
elif [ -f "code/src/main.py" ]; then
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
else
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
fi
# Create and activate virtual environment
if [ ! -d ".venv" ]; then
    python3 -m venv .venv
    echo "Virtual environment .venv created."
fi

source .venv/bin/activate
pip install --upgrade pip
pip install pandas numpy lightgbm pyarrow scikit-learn boto3 matplotlib seaborn xgboost catboost


--- 
## 2. Sync Latest Code from GitHub

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
git stash
git pull origin main


--- 
## 3. Verify Dataset Files
Ensures all 7 TSV dataset files are present in `dataset/train/` and `dataset/test/`.

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
echo "=== TRAIN DATASET ==="
ls -lh dataset/train/

echo ""
echo "=== TEST DATASET ==="
ls -lh dataset/test/

echo ""
echo "=== SQLITE INDEX DATABASES ==="
ls -lh index*.db 2>/dev/null || true


---
## 4. Run Multi-Chunk Continual Training (`--split train`)
Runs continual ensemble training across 1,600,000 Source 1 records (~70% of dataset):
- **8 sequential chunks** of 200,000 entities each (Zero-RAM leak-proof streaming).
- **68 parallel workers** fully saturating the 72 vCPUs on `ml.c5.18xlarge`.
- **80 candidates** retrieved per record to maximize Stage 1 candidate recall ($F_{0.5}$ ceiling > 0.95).
- **Fixed holdout benchmark** (5,000 entities) evaluated after each chunk to monitor true Macro $F_{0.5}$.
- **Checkpoints saved** after every chunk to `output/train/lgb_model.pkl` and `output/lgb_model.pkl`.

In [ ]:
%%bash
# Dynamic navigation to project root
if [ -f "src/main.py" ]; then
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
elif [ -f "code/src/main.py" ]; then
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
else
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
fi
source .venv/bin/activate

# Multi-chunk continual ensemble training (36 parallel workers, top-180 candidates)
python3 $MAIN \
    --split train \
    --num-chunks 4 \
    --chunk-size 40000 \
    --chunk-offset 0 \
    --max-candidates 180 \
    --val-size 5000 \
    --num-workers 36 \
    --trees-per-chunk 100


--- 
## 5. Check Training Artifacts & Model Status

In [ ]:
import os
import glob
import pickle

def find_dir(dirs):
    for d in dirs:
        if os.path.exists(d):
            return d
    return dirs[0]

print("=== Training Artifacts ===")
train_dir = find_dir(["output/train", "../output/train", "../../output/train"])
if os.path.exists(train_dir):
    for f in sorted(os.listdir(train_dir)):
        p = os.path.join(train_dir, f)
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  - {f:35s}: {size_mb:8.2f} MB")
else:
    print("  (No training directory yet. Run Section 4 to train.)")

print("\n=== Testing Outputs ===")
test_dir = find_dir(["output/test", "output", "../output", "../../output"])
if os.path.exists(test_dir):
    for f in sorted(os.listdir(test_dir)):
        if f.endswith(".tsv"):
            p = os.path.join(test_dir, f)
            size_mb = os.path.getsize(p) / (1024 * 1024)
            print(f"  - {f:35s}: {size_mb:8.2f} MB")

model_paths = ["output/train/lgb_model.pkl", "output/lgb_model.pkl", "../output/train/lgb_model.pkl", "../../output/train/lgb_model.pkl"]
model_path = next((p for p in model_paths if os.path.exists(p)), None)
if model_path:
    with open(model_path, "rb") as f:
        m = pickle.load(f)
    opt_t = m.get("optimal_threshold", 0.500)
    print(f"\nTrained Model Ready at {model_path}! Optimal Macro F0.5 Threshold: {opt_t:.3f}")


--- 
## 5a. Evaluate Upgraded High-Recall Model on Validation Set
Runs standalone validation audit on the 5,000 holdout entities using upgraded multi-signal conjunction indexing and  with 68 workers (~45 seconds).

In [ ]:
%%bash
# Dynamic navigation to project root
if [ -f "src/main.py" ]; then
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
elif [ -f "code/src/main.py" ]; then
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
else
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
fi
source .venv/bin/activate

# Fast validation evaluation on 5,000 holdout entities with upgraded conjunction indexing
python3 $MAIN \
    --split train \
    --val-size 5000 \
    --max-candidates 180 \
    --num-workers 36 \
    --no-cache \
    --eval-only


--- 
## 5b. Validation Confusion Matrix & Metrics Visualization
Loads cached validation pairs and evaluates True/False Positives/Negatives with an annotated heatmap.

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def find_file(paths):
    return next((p for p in paths if os.path.exists(p)), None)

model_file = find_file(["output/train/lgb_model.pkl", "output/lgb_model.pkl", "../output/train/lgb_model.pkl", "../../output/train/lgb_model.pkl"])
val_metrics_file = find_file(["output/train/val_metrics.pkl", "../output/train/val_metrics.pkl", "../../output/train/val_metrics.pkl"])
val_metrics = None
optimal_thresh = 0.585

if val_metrics_file:
    try:
        with open(val_metrics_file, "rb") as f:
            val_metrics = pickle.load(f)
        optimal_thresh = val_metrics.get("threshold", 0.585)
        print(f"Loaded fresh evaluation metrics from: {val_metrics_file}")
    except Exception as e:
        pass

if not val_metrics and model_file:
    try:
        with open(model_file, "rb") as f:
            model_data = pickle.load(f)
        optimal_thresh = model_data.get("optimal_threshold", 0.585)
        val_metrics = model_data.get("val_metrics", None)
        print(f"Loaded model checkpoint from: {model_file}")
    except Exception as e:
        pass

if not val_metrics:
    print("No validation metrics found yet. Run Section 4 or Section 5a to evaluate.")
else:
    tn = val_metrics.get("tn", 0)
    fp = val_metrics.get("fp", 0)
    fn = val_metrics.get("fn", 0)
    tp = val_metrics.get("tp", 0)
    prec = val_metrics.get("cand_prec", 0.0)
    rec = val_metrics.get("cand_rec", 0.0)
    cand_f05 = val_metrics.get("cand_f05", 0.0)
    blocking_rec = val_metrics.get("blocking_recall", 0.0)
    macro_f05 = val_metrics.get("macro_f05", 0.0)
    threshold = val_metrics.get("threshold", optimal_thresh)
    cm = np.array([[tn, fp], [fn, tp]])

    print("=" * 65)
    print("    OFFICIAL COMPETITION VALIDATION METRICS & CONFUSION MATRIX    ")
    print("=" * 65)
    print(f"  Optimal Decision Threshold: {threshold:.3f}")
    print(f"  Stage 1 Candidate Recall  : {blocking_rec:.2f}%")
    print(f"  Pairwise Precision        : {prec * 100:.2f}%")
    print(f"  Pairwise Recall (on cands): {rec * 100:.2f}%")
    print(f"  Pairwise F0.5 (on cands)  : {cand_f05:.4f}")
    print(f"  >>> COMPETITION MACRO F0.5: {macro_f05:.4f} <<< [OFFICIAL SCORER EQUIVALENT]")
    print("=" * 65)

    fig, ax = plt.subplots(figsize=(7, 5.5))
    labels = np.array([[
        f"True Negatives\n(TN: {tn:,})\nRate: {tn/(tn+fp):.2%}",
        f"False Positives\n(FP: {fp:,})\nRate: {fp/(tn+fp):.2%}"
    ], [
        f"False Negatives\n(FN: {fn:,})\nRate: {fn/(fn+tp):.2%}",
        f"True Positives\n(TP: {tp:,})\nRate: {tp/(fn+tp):.2%}"
    ]])
    sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=True, ax=ax,
                xticklabels=["Predicted Negative (0)", "Predicted Positive (1)"],
                yticklabels=["Actual Negative (0)", "Actual Positive (1)"])
    ax.set_title(f"Holdout Validation Confusion Matrix\nOfficial Macro F0.5 = {macro_f05:.4f} (Threshold: {threshold:.3f})", fontsize=12, fontweight="bold")
    ax.set_ylabel("Ground Truth Class", fontsize=11)
    ax.set_xlabel("Pairwise Classifier Prediction", fontsize=11)
    plt.tight_layout()
    plt.show()


---
## 6. Run Test Split Inference (`--split test`)
Executes high-throughput candidate retrieval and scoring on `test_source1.tsv` (1.73M entities):
- Uses **36 parallel workers** streaming directly from disk index (`index_test.db`).
- Uses **180 candidates** per record matching the training configuration.
- Produces official competition outputs:
  - `output/test/candidate_pairs.tsv`
  - `output/test/matching_results.tsv`


In [ ]:
%%bash
# Dynamic navigation to project root
if [ -f "src/main.py" ]; then
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
elif [ -f "code/src/main.py" ]; then
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
else
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
fi
source .venv/bin/activate

# Full test set inference (1.73M entities) using 36 parallel workers and top-180 candidates
python3 $MAIN \
    --split test \
    --sample-size 0 \
    --max-candidates 180 \
    --num-workers 36 \
    --output-dir output/


--- 
## 7. Official Submission Format Validation

In [ ]:
%%bash
# Dynamic navigation to project root
if [ -f "src/main.py" ]; then
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
elif [ -f "code/src/main.py" ]; then
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
else
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
fi
source .venv/bin/activate

# Locate test outputs and test dataset
MATCH_FILE="output/matching_results.tsv"
[ ! -f "$MATCH_FILE" ] && MATCH_FILE="output/test/matching_results.tsv"
CAND_FILE="output/candidate_pairs.tsv"
[ ! -f "$CAND_FILE" ] && CAND_FILE="output/test/candidate_pairs.tsv"

TEST_DIR="dataset/test"
[ ! -d "$TEST_DIR" ] && TEST_DIR="6ab10eb3b23ba_student_resource/student_resource/dataset/test"
[ ! -d "$TEST_DIR" ] && TEST_DIR="../dataset/test"
[ ! -d "$TEST_DIR" ] && TEST_DIR="../../dataset/test"

# Run official competition validator on outputs
python3 $VAL_SCRIPT \
    --matching "$MATCH_FILE" \
    --candidate "$CAND_FILE" \
    --test-dir "$TEST_DIR"


--- 
## 8. Preview Final Submission Files

In [ ]:
import os
import pandas as pd

def find_file(paths):
    return next((p for p in paths if os.path.exists(p)), None)

cand_file = find_file(["output/candidate_pairs.tsv", "output/test/candidate_pairs.tsv", "../output/candidate_pairs.tsv", "../../output/candidate_pairs.tsv"])
if cand_file:
    df_cand = pd.read_csv(cand_file, sep="\t", nrows=10)
    print(f"=== Candidate Pairs ({cand_file} - Top 10 Rows) ===")
    display(df_cand)
else:
    print("Candidate file not found yet. Run Section 6 to generate.")

print("\n=== Matching Results (Top 10 Rows) ===")
match_file = find_file(["output/matching_results.tsv", "output/test/matching_results.tsv", "../output/matching_results.tsv", "../../output/matching_results.tsv"])
if match_file:
    df_match = pd.read_csv(match_file, sep="\t", nrows=10)
    display(df_match)
else:
    print("Matching file not found yet. Run Section 6 to generate.")


--- 
## 9. (Optional) Sync Output Files to S3 Bucket

In [ ]:
%%bash
# Sync final outputs to your S3 bucket
aws s3 cp output/test/matching_results.tsv s3://sagemaker-eu-north-1-583664563299/amazon-ml-challenge-2026/output/matching_results.tsv
aws s3 cp output/test/candidate_pairs.tsv s3://sagemaker-eu-north-1-583664563299/amazon-ml-challenge-2026/output/candidate_pairs.tsv
echo "Outputs backed up to S3 bucket!"


---
## 9. Build Final Submission Archive (unwanted_submission.zip)
Packages `output/`, `code/business_entity_resolution/`, and `Documentation_template.md` into `unwanted_submission.zip` matching official competition specifications.

In [ ]:
%%bash
# Dynamic navigation to project root
if [ -f "src/main.py" ]; then
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
elif [ -f "code/src/main.py" ]; then
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
    MAIN="code/src/main.py"
    VAL_SCRIPT="code/src/validate_submission.py"
else
    MAIN="src/main.py"
    VAL_SCRIPT="src/validate_submission.py"
fi
source .venv/bin/activate

# Automatically sync code, copy outputs, validate, and create unwanted_submission.zip
if [ -f "package_submission.py" ]; then
    python3 package_submission.py
elif [ -f "../../package_submission.py" ]; then
    python3 ../../package_submission.py
else
    zip -r unwanted_submission.zip output code Documentation_template.md
fi
